In [ ]:
# Import Libraries
import numpy                as np
import pandas               as pd
import matplotlib.pyplot    as plt

from scipy.stats            import norm

# plt.style.use('dark_background')

# Generate Bitsream
def GenerateBitStream(length):
    stream = np.random.choice([0, 1], size=length)
    stream = ''.join(map(str, stream))
    return stream

# Generate Stochastic Stream
def GenerateStochasticStream(prob, length):
    return np.random.rand(length) < prob

# Calculate Probability
def ProbCal(bitstream):
    bs = np.array(list(bitstream), dtype=int)
    return bs.mean()

# Multiply Bitsream by AND
def MultiplyBitStreamAND(bitstream1, bitstream2):
    array1 = np.array(list(bitstream1), dtype=int)
    array2 = np.array(list(bitstream2), dtype=int)
    bitwise_and = np.bitwise_and(array1, array2)
    return ''.join(map(str, bitwise_and))

# Addition Bitstream by Multiplexer
def AdditionBitStreamMUX(bitstream1, bitstream2, select):
    array1 = np.array(list(bitstream1), dtype=int)
    array2 = np.array(list(bitstream2), dtype=int)
    sel    = np.array(list(select), dtype=int)
    bitwise_add = np.where(sel, array1, array2)
    return ''.join(map(str, bitwise_add))

# Subtraction Bitstream by Multiplexer
def SubtractionBitStreamMUX(bitstream1, bitstream2, select):
    array1 = np.array(list(bitstream1), dtype=int)
    array2 = np.array(list(bitstream2), dtype=int)
    sel    = np.array(list(select), dtype=int)
    not_arr= np.bitwise_not(array2.astype(bool))
    bitwise_add = np.where(sel, array1, not_arr)
    return ''.join(map(str, bitwise_add))

# Unipolar Multiplication
def UnipolarMultiplication(number_1, number_2, length):
    stream1     = GenerateStochasticStream(prob=number_1, length=length)
    stream2     = GenerateStochasticStream(prob=number_2, length=length)
    res         = MultiplyBitStreamAND(stream1, stream2)
    res_prob    = ProbCal(res)
    
    return res_prob

# Unipolar Addition
def UnipolarAddition(number_1, number_2, length):
    stream1     = GenerateStochasticStream(prob=number_1, length=length)
    stream2     = GenerateStochasticStream(prob=number_2, length=length)
    selStream   = GenerateStochasticStream(prob=0.5, length=length)
    res         = AdditionBitStreamMUX(stream1, stream2, selStream)
    res_prob    = ProbCal(res)
    
    return res_prob

# Unipolar Subtraction
def UnipolarSubtraction(number_1, number_2, length):
    stream1     = GenerateStochasticStream(prob=number_1, length=length)
    stream2     = GenerateStochasticStream(prob=number_2, length=length)
    selStream   = GenerateStochasticStream(prob=0.5, length=length)
    res         = SubtractionBitStreamMUX(stream1, stream2, selStream)
    res_prob    = ProbCal(res)
    
    return res_prob

# Define LFSR
def LFSR(length, signature, excluded_positions, seed=None):
    # Generate a random seed if not provided    
    if seed is None:
        seed = np.random.choice([0, 1], size=length).tolist()

    # Initialize the LFSR state with the seed
    lfsr_state = seed.copy()
    
    # List to store the output sequence
    output = []
    
    arr = np.zeros(shape=(length, length))
    
    # Generate the sequence
    for i in range(length):
        # The new bit is the XOR of the tap positions that are not excluded
        feedback_bits = [lfsr_state[tap] for tap in signature if tap not in excluded_positions]
        new_bit = np.bitwise_xor.reduce(feedback_bits)
        
        # Append the last bit of the current state to the output
        output.append(lfsr_state[-1])
        
        # print(f"Step {i+1}\t:State: {lfsr_state} -> New bit: {new_bit}")
        
        arr[i, :] = lfsr_state
        
        # Shift the register and add the new bit at position 0
        lfsr_state = [new_bit.tolist()] + lfsr_state[:-1]
    # print("-"*53)    
    
    return output, arr

# Generate Stochastic Bistream with LFSR
def generate_stochastic_bitstream(probability, lfsr_bitstream, length):
    # threshold = int(probability * len(lfsr_bitstream))
    threshold = int(probability * 2**length)
    lfsrShape = lfsr_bitstream.shape
    lfsr_lst  = list()
    for i in range(lfsrShape[0]):
        lfsr_lst.append(int(''.join(map(str, map(int, lfsr_bitstream[i, :]))), base=2))
    
    return [1 if lfsr_lst[i] < threshold else 0 for i in range(lfsrShape[0])]

# Stochastic AND
def stochastic_AND(bitstream1, bitstream2):
    return [bit1 & bit2 for bit1, bit2 in zip(bitstream1, bitstream2)]

# Stochastic Computing with LFSR
def stochastic_computing_with_lfsr(length, prob1, prob2, signature, excluded_positions, seed1=None, seed2=None):
    output1, lfsr_bitstream1 = LFSR(length, signature, excluded_positions, seed1)
    output2, lfsr_bitstream2 = LFSR(length, signature, excluded_positions, seed2)
    
    bitstream1 = generate_stochastic_bitstream(prob1, lfsr_bitstream1, length)
    bitstream2 = generate_stochastic_bitstream(prob2, lfsr_bitstream2, length)
        
    and_result = stochastic_AND(bitstream1, bitstream2)
    stochastic_result = sum(and_result) / length

    return stochastic_result, bitstream1, bitstream2, and_result

# Test Multiplication
number_1    = np.random.rand(1)
number_2    = np.random.rand(1)
length      = 1024

res         = UnipolarMultiplication(number_1=number_1, number_2=number_2, length=length)

print(f'Number 1                    -> {number_1}')
print(f'Number 2                    -> {number_2}')
print('-----------------------------')
print(f'Real Multiplication         -> {(number_1[0] * number_2[0]):.2f}')
print('-----------------------------')
print(f'Aproximation Multiplication -> {res}')

# Test Addition
number_1    = np.random.rand(1)
number_2    = np.random.rand(1)
length      = 128

res         = UnipolarAddition(number_1=number_1, number_2=number_2, length=length)

print(f'Number 1                    -> {number_1}')
print(f'Number 2                    -> {number_2}')
print('-----------------------------')
print(f'Real Addition         -> {(number_1[0] + number_2[0])/2:.2f}')
print('-----------------------------')
print(f'Aproximation Addition -> {res:.2f}')

# Test Subtraction
number_1    = np.random.rand(1)
number_2    = np.random.rand(1)
trueSub     = (number_1[0] - number_2[0] )/2 + 1/2
length      = 128

res         = UnipolarSubtraction(number_1=number_1, number_2=number_2, length=length)

print(f'Number 1                    -> {number_1}')
print(f'Number 2                    -> {number_2}')
print('-----------------------------')
print(f'Real Addition         -> {trueSub:.2f}')
print('-----------------------------')
print(f'Aproximation Addition -> {res:.2f}')

In [ ]:
# Create Dataset Multiplication
IterationNum = 1000

length       = [8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]
numel        = len(length)

number_1     = np.random.rand(IterationNum, numel)
number_2     = np.random.rand(IterationNum, numel)

dataset      = np.zeros(shape=(IterationNum, numel))

for i in range (numel):
    for j in range(IterationNum):
        res           = UnipolarMultiplication(number_1=number_1[j][i], number_2=number_2[j][i], length=length[i])
        realMult      = number_1[j][i] * number_2[j][i]
        error         = res - realMult
        dataset[j][i] = error
DataFrame = pd.DataFrame(data=dataset, columns=length)

# Show All Result with Distribution(for Multiplication)
# DataFrame.plot.hist()
# plt.show()

# Show Result Separately (for Multiplication)
# Set up subplots with a grid layout, adjust based on how many columns you have
num_columns = DataFrame.shape[1]  # Number of columns in the DataFrame
fig, axes   = plt.subplots(nrows=5, ncols=2, figsize=(20, 15))  # Adjust layout as needed
axes        = axes.flatten()  # Flatten the axes array to easily index each subplot

colors = ['#00008B', '#E9967A', '#32CD32', '#B8860B', '#9400D3', 
          '#CD5B45', '#9932CC', '#008B8B', '#FF1493', '#006400']
# Loop through each column in the DataFrame
for i, column in enumerate(DataFrame.columns):
    # Select the current axis
    ax = axes[i]

    # Plot the histogram on the subplot
    DataFrame[column].plot(kind='hist', density=True, bins=20, alpha=1, ax=ax, rwidth=0.9, color=colors[i])

    # Fit a normal distribution to the data
    mu, std = norm.fit(DataFrame[column])

    # Generate the x values for the fitted distribution
    xmin, xmax = ax.get_xlim()
    x = np.linspace(xmin, xmax, 100)
    p = norm.pdf(x, mu, std)  # PDF of the normal distribution

    # Plot the fitted normal distribution
    ax.plot(x, p, 'k', linewidth=4, label=f'$Fit: \mu={mu:.2f}, \sigma={std:.2f}$')

    # Add title and legend for each subplot
    ax.set_title(f'{column}')
    ax.legend(loc='best')

# Hide any unused subplots (if there are more subplots than DataFrame columns)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Adjust layout for better readability
plt.tight_layout()
# Save as SVG (vector format)
plt.savefig('stochastic_multiplication.svg', format='svg')

# Save as PDF (vector format)
plt.savefig('stochastic_multiplication.pdf', format='pdf')

# Save as EPS (vector format)
plt.savefig('stochastic_multiplication.eps', format='eps')

# Optionally, save as a high-quality PNG (raster format) with high DPI
plt.savefig('stochastic_multiplication.png', format='png', dpi=300)

plt.show()

In [ ]:
# Create Dataset Addition
IterationNum = 1000

length       = [8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]
numel        = len(length)

number_1     = np.random.rand(IterationNum, numel)
number_2     = np.random.rand(IterationNum, numel)

dataset      = np.zeros(shape=(IterationNum, numel))

for i in range (numel):
    for j in range(IterationNum):
        res           = UnipolarAddition(number_1=number_1[j][i], number_2=number_2[j][i], length=length[i])
        realMult      = (number_1[j][i] + number_2[j][i]) / 2
        error         = res - realMult
        dataset[j][i] = error
DataFrame = pd.DataFrame(data=dataset, columns=length)

# Show All Result (for Addition)
# DataFrame.plot.hist()
# plt.show()

# Show Result Separately with Distribution (for Addition)
# Set up subplots with a grid layout, adjust based on how many columns you have
num_columns = DataFrame.shape[1]  # Number of columns in the DataFrame
fig, axes   = plt.subplots(nrows=5, ncols=2, figsize=(20, 15))  # Adjust layout as needed
axes        = axes.flatten()  # Flatten the axes array to easily index each subplot

colors = ['#00008B', '#E9967A', '#32CD32', '#B8860B', '#9400D3', 
          '#CD5B45', '#9932CC', '#008B8B', '#FF1493', '#006400']
# Loop through each column in the DataFrame
for i, column in enumerate(DataFrame.columns):
    # Select the current axis
    ax = axes[i]

    # Plot the histogram on the subplot
    DataFrame[column].plot(kind='hist', density=True, bins=20, alpha=1, ax=ax, rwidth=0.9, color=colors[i])

    # Fit a normal distribution to the data
    mu, std = norm.fit(DataFrame[column])

    # Generate the x values for the fitted distribution
    xmin, xmax = ax.get_xlim()
    x = np.linspace(xmin, xmax, 100)
    p = norm.pdf(x, mu, std)  # PDF of the normal distribution

    # Plot the fitted normal distribution
    ax.plot(x, p, 'k', linewidth=4, label=f'$Fit: \mu={mu:.2f}, \sigma={std:.2f}$')

    # Add title and legend for each subplot
    ax.set_title(f'{column}')
    ax.legend(loc='best')

# Hide any unused subplots (if there are more subplots than DataFrame columns)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Adjust layout for better readability
plt.tight_layout()

# Save as SVG (vector format)
plt.savefig('stochastic_addition.svg', format='svg')

# Save as PDF (vector format)
plt.savefig('stochastic_addition.pdf', format='pdf')

# Save as EPS (vector format)
plt.savefig('stochastic_addition.eps', format='eps')

# Optionally, save as a high-quality PNG (raster format) with high DPI
plt.savefig('stochastic_addition.png', format='png', dpi=300)

plt.show()

In [ ]:
# Create Dataset Subtraction
IterationNum = 1000

length       = [8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]
numel        = len(length)

number_1     = np.random.rand(IterationNum, numel)
number_2     = np.random.rand(IterationNum, numel)

dataset      = np.zeros(shape=(IterationNum, numel))

for i in range (numel):
    for j in range(IterationNum):
        res           = UnipolarSubtraction(number_1=number_1[j][i], number_2=number_2[j][i], length=length[i])
        realMult      = (number_1[j][i] - number_2[j][i]) / 2 + 1/2
        error         = res - realMult
        dataset[j][i] = error
DataFrame = pd.DataFrame(data=dataset, columns=length)

# # Show All Result (for Subtraction)
# DataFrame.plot.hist()
# plt.show()

# Show All Result Separately with Distribution (for Subtraction)
# Set up subplots with a grid layout, adjust based on how many columns you have
num_columns = DataFrame.shape[1]  # Number of columns in the DataFrame
fig, axes   = plt.subplots(nrows=5, ncols=2, figsize=(20, 15))  # Adjust layout as needed
axes        = axes.flatten()  # Flatten the axes array to easily index each subplot

colors = ['#00008B', '#E9967A', '#32CD32', '#B8860B', '#9400D3', 
          '#CD5B45', '#9932CC', '#008B8B', '#FF1493', '#006400']
# Loop through each column in the DataFrame
for i, column in enumerate(DataFrame.columns):
    # Select the current axis
    ax = axes[i]

    # Plot the histogram on the subplot
    DataFrame[column].plot(kind='hist', density=True, bins=20, alpha=1, ax=ax, rwidth=0.9, color=colors[i])

    # Fit a normal distribution to the data
    mu, std = norm.fit(DataFrame[column])

    # Generate the x values for the fitted distribution
    xmin, xmax = ax.get_xlim()
    x = np.linspace(xmin, xmax, 100)
    p = norm.pdf(x, mu, std)  # PDF of the normal distribution

    # Plot the fitted normal distribution
    ax.plot(x, p, 'k', linewidth=4, label=f'$Fit: \mu={mu:.2f}, \sigma={std:.2f}$')

    # Add title and legend for each subplot
    ax.set_title(f'{column}')
    ax.legend(loc='best')

# Hide any unused subplots (if there are more subplots than DataFrame columns)
# for j in range(i + 1, len(axes)):
#     fig.delaxes(axes[j])

# Adjust layout for better readability
# plt.tight_layout()
# plt.show()

# Test LFSR
# Example parameters
# length              = 128  
# prob1               = 0.2  
# prob2               = 0.7
# randnum             = np.random.choice(range(length), length)
# signature           = randnum[:(length-2)]
# excluded_positions  = randnum[(length-2):]
# seed1               = [1, 0, 1, 0, 1, 0]
# seed2               = [0, 1, 1, 1, 0, 0]  

# iterNum             = 1

# arr                 = np.zeros(shape=(iterNum, 1))

# for i in range(iterNum):
#     # Perform stochastic computing
#     stochastic_result, bitstream1, bitstream2, and_result = stochastic_computing_with_lfsr(
#         length, prob1, prob2, signature, excluded_positions, None, None
#     )

# stochastic_result, bitstream1, bitstream2, and_result = stochastic_computing_with_lfsr(
#     length, prob1, prob2, signature, excluded_positions, None, None)
    
# # True result
# true_result = prob1 * prob2

#     # print("Stochastic result:", stochastic_result)
#     # print("True result      :", true_result)
#     # print('-------------------------------------')
# df = pd.DataFrame([stochastic_result], index=['prefix ' + str(i) for i in range(iterNum)])
# df

# Create Dataset
# IterationNum        = 1000

# length              = [8, 16, 32, 64, 128, 256]
# numel               = len(length)

# prob1               = np.random.rand(IterationNum, numel)
# prob2               = np.random.rand(IterationNum, numel)

# dataset             = np.zeros(shape=(IterationNum, numel))

# iterNum             = 1

# for i in range (numel):
#     print(length[i])
#     for j in range(IterationNum):
#         randnum             = np.random.choice(range(length[i]), length[i])
#         signature           = randnum[:(length[i]-2)]
#         excluded_positions  = randnum[(length[i]-2):]
#         stochastic_result, bitstream1, bitstream2, and_result = stochastic_computing_with_lfsr(
#             length[i], prob1[j][i], prob2[j][i], signature, excluded_positions, None, None)
#         realMult      = prob1[j][i] * prob2[j][i]
#         error         = stochastic_result - realMult
#         dataset[j][i] = error
# DataFrame = pd.DataFrame(data=dataset, columns=length)
# #################################################################################################
# xmin        = DataFrame.min().min()
# xmax        = DataFrame.max().max()
# mu          = DataFrame.mean()
# std         = DataFrame.std()
# length      = 500
# columns     = len(mu)
# columnsName = [8, 16, 32, 64, 128, 256]

# x = np.linspace(xmin, xmax, length)
# df = pd.DataFrame(np.zeros(shape=(length, columns)))
# for i, (m, s) in enumerate(zip(mu, std)):
#     p = norm.pdf(x, m, s)
#     df.iloc[:, i] = p
# df.columns = columnsName

# # Show Result with Plot Distribution
# # Set up subplots with a grid layout, adjust based on how many columns you have
# num_columns = DataFrame.shape[1]  # Number of columns in the DataFrame
# fig, axes   = plt.subplots(nrows=5, ncols=2, figsize=(20, 15))  # Adjust layout as needed
# axes        = axes.flatten()  # Flatten the axes array to easily index each subplot

# colors = ['darkblue', 'darkred', 'darkgreen', 'darkorange', 'purple', 'brown', 'black', 'darkslategray']

# # Loop through each column in the DataFrame
# for i, column in enumerate(DataFrame.columns):
#     # Select the current axis
#     ax = axes[i]

#     # Plot the histogram on the subplot
#     DataFrame[column].plot(kind='hist', density=True, bins=20, alpha=0.9, ax=ax, rwidth=0.9, color=colors[i])

#     # Fit a normal distribution to the data
#     mu, std = norm.fit(DataFrame[column])

#     # Generate the x values for the fitted distribution
#     xmin, xmax = ax.get_xlim()
#     x = np.linspace(xmin, xmax, 100)
#     p = norm.pdf(x, mu, std)  # PDF of the normal distribution

#     # Plot the fitted normal distribution
#     ax.plot(x, p, 'k', linewidth=4, label=f'Fit: $\mu={mu:.2f}, \sigma={std:.2f}$')

#     # Add title and legend for each subplot
#     ax.set_title(f'{column}')
#     ax.legend(loc='best')

# # Hide any unused subplots (if there are more subplots than DataFrame columns)
# for j in range(i + 1, len(axes)):
#     fig.delaxes(axes[j])

# Adjust layout for better readability
plt.tight_layout()

# Save as SVG (vector format)
plt.savefig('stochastic_subtraction.svg', format='svg')

# Save as PDF (vector format)
plt.savefig('stochastic_subtraction.pdf', format='pdf')

# Save as EPS (vector format)
plt.savefig('stochastic_subtraction.eps', format='eps')

# Optionally, save as a high-quality PNG (raster format) with high DPI
plt.savefig('stochastic_subtraction.png', format='png', dpi=300)

plt.show()